# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
import pandas as pd

frame = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
cols = ["impressions_90d", "avg_position", "ctr", "engagement_rate", "days_since_last_update"]
desc = frame[cols].describe(percentiles=[0.5, 0.9, 0.99]).T
desc


,count,mean,std,min,50%,90%,99%,max
impressions_90d,30000.0,5200.366300,16838.019547,1.0,731.00,12136.40,73505.830,517715.0
avg_position,30000.0,16.342380,15.216790,0.0,10.80,36.80,69.901,245.0
ctr,30000.0,0.510733,3.279162,0.0,0.07,0.65,8.330,100.0
engagement_rate,30000.0,2.534520,8.310096,0.0,0.00,6.94,33.330,100.0
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.00,104.00,106.000,373.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

- **Test 1 — declining pages have lower CTR than stable pages.**
- **Test 2 — stale pages (long since last update) decline more often than fresh pages.**
- **Test 3 — pages with any AI-referral traffic (`ai_traffic_pct`) behave differently from those with none.**

In [2]:
declining = frame[frame["trend_direction"] == "down"]
stable = frame[frame["trend_direction"] != "down"]

print("Test 1 — CTR, declining vs stable")
print(f"  declining mean CTR: {declining['ctr'].mean():.2f}   stable mean CTR: {stable['ctr'].mean():.2f}")
verdict1 = "CONFIRMED" if declining["ctr"].mean() < stable["ctr"].mean() else "OPPOSITE"
print(f"  Verdict: {verdict1}")
print()

print("Test 2 — staleness, declining vs stable")
d_med, s_med = declining["days_since_last_update"].median(), stable["days_since_last_update"].median()
d_mean, s_mean = declining["days_since_last_update"].mean(), stable["days_since_last_update"].mean()
print(f"  median days_since_last_update  - declining: {d_med:.0f}   stable: {s_med:.0f}")
print(f"  mean days_since_last_update    - declining: {d_mean:.1f}  stable: {s_mean:.1f}")
verdict2 = "MIXED" if d_med == s_med else ("CONFIRMED" if d_med > s_med else "OPPOSITE")
print(f"  Verdict: {verdict2} — medians are tied, but the mean gap (heavy right tail) leans the expected direction")
print()

print("Test 3 — AI-referral share, declining vs stable")
has_ai = frame[frame["ai_traffic_pct"] > 0]
print(f"  pages with any AI-referral share: {len(has_ai):,} of {len(frame):,} ({len(has_ai)/len(frame):.1%})")
print(f"  declining mean ai_traffic_pct: {declining['ai_traffic_pct'].mean():.2f}"
      f"   stable: {stable['ai_traffic_pct'].mean():.2f}")
print("  Verdict: MIXED — AI-referral sessions are too sparse in this slice to call directionally; treat as EDA, not a ranking feature.")


Test 1 — CTR, declining vs stable
  declining mean CTR: 0.32   stable mean CTR: 0.73
  Verdict: CONFIRMED

Test 2 — staleness, declining vs stable
  median days_since_last_update  - declining: 20   stable: 20
  mean days_since_last_update    - declining: 49.2  stable: 42.4
  Verdict: MIXED — medians are tied, but the mean gap (heavy right tail) leans the expected direction

Test 3 — AI-referral share, declining vs stable
  pages with any AI-referral share: 1,930 of 30,000 (6.4%)
  declining mean ai_traffic_pct: 0.78   stable: 0.75
  Verdict: MIXED — AI-referral sessions are too sparse in this slice to call directionally; treat as EDA, not a ranking feature.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

The baseline rule's `low_ctr_visible_page` flag assumes: **high impressions + decent average position + low CTR together mark an underperforming page worth reviewing.** Testing that assumption directly against the decline label.

In [3]:
flagged = frame[
    (frame["impressions_90d"] >= 500)
    & (frame["avg_position"] > 0) & (frame["avg_position"] <= 20)
    & (frame["ctr"] < 0.5)
]
rest = frame.drop(flagged.index)

flagged_decline_rate = (flagged["trend_direction"] == "down").mean()
rest_decline_rate = (rest["trend_direction"] == "down").mean()

print(f"Pages matching low_ctr_visible_page conditions: {len(flagged):,}")
print(f"Decline rate inside the flag : {flagged_decline_rate:.1%}")
print(f"Decline rate outside the flag: {rest_decline_rate:.1%}")
verdict = "CONFIRMED" if flagged_decline_rate > rest_decline_rate else "MIXED"
note = "holds up" if verdict == "CONFIRMED" else "is not clearly supported"
print(f"Verdict: {verdict} - the flags assumption {note} on this data.")


Pages matching low_ctr_visible_page conditions: 9,759
Decline rate inside the flag : 62.7%
Decline rate outside the flag: 50.1%
Verdict: CONFIRMED - the flags assumption holds up on this data.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The CTR-at-visibility signal is clearly confirmed and safe to keep leaning on for prioritization. Staleness is murkier than the baseline rule assumes: the *typical* declining page is no staler than a typical stable one (identical medians), and only the *mean* leans the expected way, dragged up by a heavy-tailed group of very old pages — so `days_since_last_update` is a weaker signal on its own than the rule treats it, and works best combined with other conditions rather than trusted alone. The AI-referral signal is too sparse in this slice to trust for ranking yet — worth watching and re-testing as more AI-referral traffic accumulates, not a scoring input today.

In [4]:
print("Keep leaning on : CTR-at-visibility — clearly confirmed on this data")
print("Use with caution: staleness alone — medians tied, only the mean (heavy tail) leans the expected way")
print("Watch, do not rank on yet: AI-referral share - too sparse to trust directionally")


Keep leaning on : CTR-at-visibility — clearly confirmed on this data
Use with caution: staleness alone — medians tied, only the mean (heavy tail) leans the expected way
Watch, do not rank on yet: AI-referral share - too sparse to trust directionally


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.